# 02 — U-Net dari Nol: Segmentasi Hewan Peliharaan

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riki-profile/ai-engineer-journey/blob/main/04-segmentation/02_unet_dari_nol.ipynb)

**U-Net** (Ronneberger et al., 2015) adalah arsitektur *encoder–decoder* yang menjadi fondasi banyak model segmentasi modern, dari citra medis hingga *diffusion model*.

Isi notebook:
1. Dataset **Oxford-IIIT Pet**: foto kucing/anjing dengan mask 3 kelas (hewan, background, tepi).
2. Transformasi yang **disinkronkan** antara gambar dan mask.
3. Membangun U-Net dari nol, termasuk melacak ukuran tensor di encoder, decoder, dan *skip connection*.
4. Loss gabungan **Cross-Entropy + Dice**.
5. Training dengan metrik **mIoU** dan **Dice**, lalu visualisasi prediksi.

> Notebook ini dirancang untuk **Google Colab dengan GPU**.

## 0. Cek GPU

Di Colab pilih **Runtime → Change runtime type → T4 GPU**, lalu jalankan ulang dari atas. Notebook tetap bisa berjalan di CPU, tetapi jauh lebih lambat.

In [ ]:
import shutil
import subprocess
import torch

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU terdeteksi: {props.name} | memori {props.total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU tidak terdeteksi, akan berjalan di CPU (lebih lambat).")
    print("   Di Colab: Runtime → Change runtime type → T4 GPU")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"])

## 1. Setup dan Konfigurasi

`QUICK_RUN=1` (environment variable) memakai subset sangat kecil dan 1 epoch untuk uji cepat di CPU.

`IMG_SIZE = 128` dipilih agar training cepat di Colab. Resolusi lebih tinggi menghasilkan tepi lebih halus dengan biaya waktu lebih lama.

In [ ]:
import os
import time
import random
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

torch.manual_seed(42)
random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

QUICK_RUN = os.environ.get("QUICK_RUN") == "1"
IMG_SIZE = 128
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 1 if QUICK_RUN else 20
DATA_DIR = "./data"
NUM_WORKERS = 2

KELAS = ["hewan", "background", "tepi"]
N_KELAS = len(KELAS)
print("device:", device, "| QUICK_RUN:", QUICK_RUN, "| EPOCHS:", EPOCHS)

## 2. Dataset Oxford-IIIT Pet

`torchvision.datasets.OxfordIIITPet` dengan `target_types="segmentation"` mengembalikan pasangan (gambar, *trimap*). Nilai piksel trimap:
- `1` = hewan
- `2` = background
- `3` = tepi (area batas yang ambigu)

Kita kurangi 1 agar menjadi indeks kelas `0, 1, 2` yang cocok untuk `CrossEntropyLoss`. Unduhan pertama sekitar 800 MB.

In [ ]:
raw_trainval = datasets.OxfordIIITPet(DATA_DIR, split="trainval", target_types="segmentation", download=True)
raw_test = datasets.OxfordIIITPet(DATA_DIR, split="test", target_types="segmentation", download=True)
print("trainval:", len(raw_trainval), "| test:", len(raw_test))

img, trimap = raw_trainval[0]
print("ukuran gambar:", img.size, "| nilai unik trimap:", np.unique(np.array(trimap)))

## 3. Transformasi yang Disinkronkan

Pada klasifikasi, augmentasi hanya mengubah gambar. Pada segmentasi, **mask harus mengalami transformasi geometris yang sama persis**: jika gambar dibalik, mask juga harus dibalik. Karena itu kita tidak bisa memakai `transforms.Compose` biasa, dan menulis transformasi sendiri dengan `torchvision.transforms.functional`.

Aturan penting saat *resize* mask: gunakan interpolasi **NEAREST**. Interpolasi bilinear akan menghasilkan nilai "di antara" kelas (misalnya 1.5), yang tidak bermakna sebagai label.

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


class PetSegmentation(Dataset):
    def __init__(self, base, augment=False):
        self.base, self.augment = base, augment

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        img, mask = self.base[i]
        img = TF.resize(img, [IMG_SIZE, IMG_SIZE], interpolation=InterpolationMode.BILINEAR)
        mask = TF.resize(mask, [IMG_SIZE, IMG_SIZE], interpolation=InterpolationMode.NEAREST)
        if self.augment:
            if random.random() < 0.5:           # flip yang SAMA untuk gambar dan mask
                img, mask = TF.hflip(img), TF.hflip(mask)
            img = TF.adjust_brightness(img, random.uniform(0.8, 1.2))  # hanya gambar (bukan geometris)
        img = TF.normalize(TF.to_tensor(img), MEAN, STD)
        mask = torch.as_tensor(np.array(mask), dtype=torch.long) - 1   # 1,2,3 -> 0,1,2
        return img, mask


# Split trainval → train (90%) dan val (10%); test terpisah
idx = torch.randperm(len(raw_trainval), generator=torch.Generator().manual_seed(42)).tolist()
n_val = len(idx) // 10
val_idx, train_idx = idx[:n_val], idx[n_val:]
test_idx = list(range(len(raw_test)))
if QUICK_RUN:
    train_idx, val_idx, test_idx = train_idx[:64], val_idx[:32], test_idx[:32]

train_set = Subset(PetSegmentation(raw_trainval, augment=True), train_idx)
val_set = Subset(PetSegmentation(raw_trainval, augment=False), val_idx)
test_set = Subset(PetSegmentation(raw_test, augment=False), test_idx)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(device == "cuda"))
train_loader = DataLoader(train_set, shuffle=True, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)
print(f"train: {len(train_set)} | val: {len(val_set)} | test: {len(test_set)}")

Visualisasi satu batch: gambar, mask, dan *overlay*. Perhatikan area "tepi" (kuning) yang tipis di sekeliling hewan.

In [ ]:
def denormalize(img):
    return (img * torch.tensor(STD).view(3, 1, 1) + torch.tensor(MEAN).view(3, 1, 1)).clamp(0, 1).permute(1, 2, 0)

CMAP = plt.matplotlib.colors.ListedColormap(["#d6604d", "#4393c3", "#fee08b"])  # hewan, background, tepi

images, masks = next(iter(train_loader))
print("images:", tuple(images.shape), "| masks:", tuple(masks.shape), masks.dtype, "| kelas:", masks.unique().tolist())

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
for j in range(6):
    axes[0, j].imshow(denormalize(images[j]))
    axes[1, j].imshow(masks[j], cmap=CMAP, vmin=0, vmax=2)
    axes[2, j].imshow(denormalize(images[j])); axes[2, j].imshow(masks[j] == 0, cmap="Reds", alpha=0.4)
for a, judul in zip(axes[:, 0], ["gambar", "mask (merah=hewan)", "overlay hewan"]):
    a.set_ylabel(judul)
for a in axes.flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 4. Arsitektur U-Net

U-Net berbentuk huruf **U**:

```
input 3×128×128
  │ DoubleConv ─────────────────────────────── skip ──────────────────┐
  ▼ Down (MaxPool + DoubleConv)  → 64×64                               │
  │ ────────────────────────────────── skip ───────────────┐           │
  ▼ Down → 32×32                                           │           │
  │ ──────────────────────── skip ───────┐                 │           │
  ▼ Down → 16×16                         │                 │           │
  │ ──────────── skip ──┐                │                 │           │
  ▼ Down → 8×8 (bottleneck)              │                 │           │
  ▲ Up → 16×16 + concat ◄┘               │                 │           │
  ▲ Up → 32×32 + concat ◄────────────────┘                 │           │
  ▲ Up → 64×64 + concat ◄──────────────────────────────────┘           │
  ▲ Up → 128×128 + concat ◄────────────────────────────────────────────┘
  │ Conv 1×1 → N_KELAS×128×128 (logits per piksel)
```

- **Encoder** (turun): seperti CNN klasifikasi. Resolusi mengecil, channel bertambah, fitur makin abstrak ("apa").
- **Decoder** (naik): `ConvTranspose2d` memperbesar resolusi kembali.
- **Skip connection**: fitur encoder beresolusi tinggi **digabung (concat)** ke decoder. Informasi detail lokasi ("di mana") yang hilang saat downsampling dipulihkan dari sini. Inilah kunci tepi mask yang tajam.
- Output: logits `(N_KELAS, H, W)`, satu skor per kelas per piksel.

In [ ]:
class DoubleConv(nn.Module):
    # (Conv 3×3 → BatchNorm → ReLU) × 2, ukuran spasial tetap
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Down(nn.Module):
    # MaxPool (ukuran /2) lalu DoubleConv
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))

    def forward(self, x):
        return self.net(x)


class Up(nn.Module):
    # ConvTranspose (ukuran ×2), gabungkan dengan skip, lalu DoubleConv
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch // 2, kernel_size=2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)   # in_ch = (in_ch // 2 dari up) + (in_ch // 2 dari skip)

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)          # concat di dimensi channel
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, n_kelas=3, base=32):
        super().__init__()
        c1, c2, c3, c4, c5 = base, base * 2, base * 4, base * 8, base * 16
        self.inc = DoubleConv(in_ch, c1)
        self.down1, self.down2, self.down3, self.down4 = Down(c1, c2), Down(c2, c3), Down(c3, c4), Down(c4, c5)
        self.up1, self.up2, self.up3, self.up4 = Up(c5, c4), Up(c4, c3), Up(c3, c2), Up(c2, c1)
        self.outc = nn.Conv2d(c1, n_kelas, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)          # bottleneck
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)


model = UNet(n_kelas=N_KELAS).to(device)
print(f"Jumlah parameter: {sum(p.numel() for p in model.parameters()):,}")

### 4.1 Melacak ukuran tensor

*Forward hook* dipasang di setiap blok untuk mencetak ukuran output-nya. Perhatikan simetri encoder–decoder: ukuran output `up1` sama dengan `down3`, `up2` sama dengan `down2`, dan seterusnya, sehingga skip connection bisa digabung.

In [ ]:
ukuran = []
hooks = [getattr(model, nama).register_forward_hook(
             lambda m, i, o, nama=nama: ukuran.append((nama, tuple(o.shape))))
         for nama in ["inc", "down1", "down2", "down3", "down4", "up1", "up2", "up3", "up4", "outc"]]

model.eval()
with torch.no_grad():
    model(torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device))
for h in hooks:
    h.remove()

for nama, s in ukuran:
    peran = "encoder" if nama.startswith(("inc", "down")) else ("output" if nama == "outc" else "decoder")
    print(f"{nama:<6} {peran:<8} {str(s)}")

## 5. Loss: Cross-Entropy + Dice

- **Cross-Entropy** per piksel: stabil dan mudah dioptimasi, tetapi setiap piksel berbobot sama, sehingga kelas kecil (area "tepi" yang tipis) mudah terabaikan.
- **Soft Dice loss** = `1 − Dice`, dihitung dari **probabilitas** (softmax), bukan argmax, agar bisa diturunkan (*differentiable*). Dice menilai tumpang tindih per kelas, sehingga kelas kecil ikut diperhatikan.

Kombinasi keduanya adalah pilihan umum dan kuat untuk segmentasi.

In [ ]:
def soft_dice_loss(logits, target, eps=1e-6):
    probs = torch.softmax(logits, dim=1)                                    # (N, C, H, W)
    one_hot = F.one_hot(target, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)                                                         # jumlahkan per kelas
    irisan = (probs * one_hot).sum(dims)
    total = probs.sum(dims) + one_hot.sum(dims)
    dice = (2 * irisan + eps) / (total + eps)
    return 1 - dice.mean()


ce_loss = nn.CrossEntropyLoss()

def loss_fn(logits, target):
    return ce_loss(logits, target) + soft_dice_loss(logits, target)


# Uji cepat: prediksi sempurna → Dice loss mendekati 0
t = torch.randint(0, N_KELAS, (2, 8, 8))
logit_sempurna = F.one_hot(t, N_KELAS).permute(0, 3, 1, 2).float() * 20
print(f"Dice loss prediksi sempurna: {soft_dice_loss(logit_sempurna, t):.4f}")
print(f"Dice loss prediksi acak    : {soft_dice_loss(torch.randn(2, N_KELAS, 8, 8), t):.4f}")

## 6. Training dan Evaluasi

Metrik dihitung dari **confusion matrix piksel** yang diakumulasi selama satu epoch penuh (seperti notebook 01), bukan dirata-rata per batch, agar hasilnya tepat.

`torch.bincount` menghitung confusion matrix langsung di GPU.

In [ ]:
def update_cm(cm, logits, target):
    pred = logits.argmax(dim=1)
    idx = target.view(-1) * N_KELAS + pred.view(-1)
    cm += torch.bincount(idx, minlength=N_KELAS ** 2).view(N_KELAS, N_KELAS)
    return cm


def metrik(cm):
    cm = cm.double()
    tp = cm.diag()
    fp, fn = cm.sum(0) - tp, cm.sum(1) - tp
    iou = tp / (tp + fp + fn).clamp(min=1)
    dice = 2 * tp / (2 * tp + fp + fn).clamp(min=1)
    return {"pixel_acc": (tp.sum() / cm.sum()).item(), "iou": iou.cpu().numpy(),
            "miou": iou.mean().item(), "dice": dice.mean().item()}


def run_epoch(model, loader, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, n = 0.0, 0
    cm = torch.zeros(N_KELAS, N_KELAS, dtype=torch.long, device=device)
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            if train:
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
            total_loss += loss.item() * x.size(0); n += x.size(0)
            update_cm(cm, logits.detach(), y)
    return total_loss / n, metrik(cm)


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

Training loop. Bobot dengan **mIoU validasi terbaik** disimpan (dalam memori). Pada GPU T4, satu epoch penuh butuh sekitar 20–40 detik.

In [ ]:
import copy

history = {"train_loss": [], "val_loss": [], "train_miou": [], "val_miou": []}
best_miou, best_state = -1.0, None
mulai = time.time()

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_m = run_epoch(model, train_loader, optimizer)
    va_loss, va_m = run_epoch(model, val_loader)
    scheduler.step()
    for k, v in zip(history, [tr_loss, va_loss, tr_m["miou"], va_m["miou"]]):
        history[k].append(v)
    if va_m["miou"] > best_miou:
        best_miou, best_state = va_m["miou"], copy.deepcopy(model.state_dict())
    print(f"Epoch {epoch:2d}/{EPOCHS} | train loss {tr_loss:.3f} mIoU {tr_m['miou']:.3f} | "
          f"val loss {va_loss:.3f} mIoU {va_m['miou']:.3f} dice {va_m['dice']:.3f}")

print(f"\nWaktu: {time.time() - mulai:.0f} detik | mIoU val terbaik: {best_miou:.3f}")
model.load_state_dict(best_state)

## 7. Kurva Training

In [ ]:
ep = range(1, EPOCHS + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, history["train_loss"], "o-", label="train"); ax[0].plot(ep, history["val_loss"], "o-", label="val")
ax[1].plot(ep, history["train_miou"], "o-", label="train"); ax[1].plot(ep, history["val_miou"], "o-", label="val")
for a, judul in zip(ax, ["Loss (CE + Dice)", "mIoU"]):
    a.set_title(judul); a.set_xlabel("epoch"); a.legend()
plt.tight_layout(); plt.show()

## 8. Evaluasi di Data Test

IoU per kelas biasanya menunjukkan pola: **background** dan **hewan** tinggi, sedangkan **tepi** jauh lebih rendah karena areanya tipis dan memang ambigu.

In [ ]:
test_loss, test_m = run_epoch(model, test_loader)
print(f"Test loss {test_loss:.3f} | pixel acc {test_m['pixel_acc']:.3f} | mIoU {test_m['miou']:.3f} | Dice {test_m['dice']:.3f}\n")
for k, v in zip(KELAS, test_m["iou"]):
    print(f"IoU {k:<11} {v:.3f}  {'█' * int(v * 40)}")

## 9. Visualisasi Prediksi

Baris: gambar, mask asli, prediksi, dan peta kesalahan (merah = piksel salah). Kesalahan biasanya terkonsentrasi di tepi objek.

In [ ]:
model.eval()
x, y = next(iter(test_loader))
with torch.no_grad():
    pred = model(x.to(device)).argmax(1).cpu()

n = min(6, len(x))
fig, axes = plt.subplots(4, n, figsize=(2.3 * n, 9.5))
for j in range(n):
    axes[0, j].imshow(denormalize(x[j]))
    axes[1, j].imshow(y[j], cmap=CMAP, vmin=0, vmax=2)
    axes[2, j].imshow(pred[j], cmap=CMAP, vmin=0, vmax=2)
    axes[3, j].imshow(pred[j] != y[j], cmap="Reds")
    iou_hewan = ((pred[j] == 0) & (y[j] == 0)).sum() / (((pred[j] == 0) | (y[j] == 0)).sum()).clamp(min=1)
    axes[0, j].set_title(f"IoU hewan {iou_hewan:.2f}", fontsize=9)
for a, judul in zip(axes[:, 0], ["gambar", "asli", "prediksi", "salah"]):
    a.set_ylabel(judul)
for a in axes.flat:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Ringkasan

- Augmentasi geometris harus diterapkan **sama persis** ke gambar dan mask; resize mask memakai **NEAREST**.
- U-Net = encoder (apa) + decoder (di mana) + **skip connection** yang memulihkan detail spasial.
- Output model `(N, C, H, W)`; label `(N, H, W)` berisi indeks kelas → cocok langsung dengan `CrossEntropyLoss`.
- **CE + soft Dice** menyeimbangkan stabilitas training dan perhatian ke kelas kecil.
- Evaluasi dengan **mIoU** dari confusion matrix yang diakumulasi di seluruh dataset.

## Latihan

1. **Ablasi skip connection.** Buat varian `UNetTanpaSkip` yang tidak melakukan concat (sesuaikan jumlah channel di `Up`). Latih dengan jumlah epoch yang sama dan bandingkan mIoU serta visualisasi tepi objek. Seberapa besar peran skip connection?
2. **Pengaruh loss.** Latih tiga model dengan loss berbeda: CE saja, Dice saja, dan CE + Dice. Buat tabel IoU per kelas (terutama kelas **tepi**) dan jelaskan mengapa hasilnya berbeda.
3. **Encoder pretrained.** Ganti encoder dengan ResNet18 pretrained ImageNet (ambil `conv1…layer4` dari `torchvision.models.resnet18`) dan bangun decoder dengan skip dari output tiap `layer`. Bandingkan mIoU dan kecepatan konvergensi dengan U-Net dari nol (hubungkan dengan transfer learning di modul 02).